# Fine-Tuning: BGE-M3 on TripLegal-CL

**Objective.** The central claim of the paper is that TripLegal-CL provides
an effective contrastive training signal for adapting dense bi-encoder
retrievers to the Spanish legal domain. To validate this, we fine-tune
`BAAI/bge-m3` — a strong multilingual encoder optimized for dense
retrieval and reranking — using contrastive learning on TripLegal-CL.
If the fine-tuned model consistently outperforms its baseline across
all IR metrics, this confirms that the corpus provides useful
domain-specific supervision.

**Dev evaluator.** During training, a dev evaluator (50K queries, 80K
corpus) runs every 100 steps to monitor convergence. It is drawn from
the **training region** (first 380K instances) and is intentionally
smaller for speed.

## 1. Environment Setup

In [1]:
!pip install -U "sentence-transformers>=3.0.1" "transformers>=4.48.0" datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## 2. Imports

In [2]:
import logging
import traceback

from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

## 3. Load Model

In [3]:
model = SentenceTransformer(
    "BAAI/bge-m3",
    model_card_data=SentenceTransformerModelCardData(
        language="es",
        license="apache-2.0",
        model_name="BGE-M3 trained on TripLegal-CL Legal Spanish.",
    ),
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

**Note:** Unlike multilingual-e5-large, BGE-M3 does **not** require
"query: " / "passage: " prefixes. Raw text is passed directly to
the encoder — no `prompts={}` parameter is needed in the training
arguments.

## 4. Prepare Data Helper

In [4]:
def select_first_pos(example):
    if example["pos"]:
        return {"query": example["query"], "pos": example["pos"][0]}

## 5. Load and Split Dataset

In [5]:
SEED = 42
TRAIN_N = 300_000
EVAL_N  = 50_000
TEST_N  = 30_000

base = load_dataset("wilfredomartel/TripLegal-CL", split="train").shuffle(seed=SEED)

# Disjoint ranges — no data leakage
train_dataset = base.select(range(0, TRAIN_N))
eval_dataset  = base.select(range(TRAIN_N, TRAIN_N + EVAL_N))
test_dataset  = base.select(range(TRAIN_N + EVAL_N, TRAIN_N + EVAL_N + TEST_N))

cols_to_remove = ["neg", "pos_score", "neg_score"]

train_dataset = train_dataset.remove_columns(cols_to_remove).map(select_first_pos)
eval_dataset  = eval_dataset.remove_columns(cols_to_remove).map(select_first_pos)
test_dataset  = test_dataset.remove_columns(cols_to_remove).map(select_first_pos)

print(f"Train: {len(train_dataset):,}")
print(f"Eval:  {len(eval_dataset):,}")
print(f"Test:  {len(test_dataset):,}")
print(train_dataset[0])

README.md:   0%|          | 0.00/428 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/592382 [00:00<?, ? examples/s]

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Train: 300,000
Eval:  50,000
Test:  30,000
{'query': '¿Por qué la Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión?', 'pos': 'La Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión, al determinar que los agravios presentados por Manuel Contreras Ramos no combatían las razones del acuerdo de presidencia recurrido. El acuerdo de desechamiento se basó en la inexistencia de una cuestión propiamente constitucional, mientras que los agravios del recurrente se enfocaron en demostrar la importancia y trascendencia del asunto, sin desvirtuar la falta de un tema de constitucionalidad. La Sala aplicó la tesis 1a. XXXVI/2018 (10a.) para señalar que los agravios que no desvirtúan la inexistencia de una cuestión constitucional son inoperantes, y que la falta de un tema de co

## 6. Loss Function and Training Arguments

In [6]:
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)

run_name = "bge-m3-tripLegalCL-300k"

args = SentenceTransformerTrainingArguments(
    output_dir=f"models/{run_name}",
    num_train_epochs=1,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # BGE-M3 does NOT use prompts — no prefixes needed
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=20,
    run_name=run_name,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### How contrastive learning works in this training setup

> **Sources:** This explanation is based on the official `sentence-transformers`
> documentation and source code:
> - [MultipleNegativesRankingLoss — Losses documentation](https://sbert.net/docs/package_reference/sentence_transformer/losses.html)
> - [CachedMultipleNegativesRankingLoss — source code](https://github.com/huggingface/sentence-transformers/blob/main/sentence_transformers/losses/CachedMultipleNegativesRankingLoss.py)
> - [NoDuplicatesBatchSampler — Samplers documentation](https://sbert.net/docs/package_reference/sentence_transformer/sampler.html)
> - [Loss Overview — sentence-transformers](https://sbert.net/docs/sentence_transformer/loss_overview.html)
> - Original paper: *Efficient Natural Language Response Suggestion for Smart Reply*, Section 4.4 ([Henderson et al., 2017](https://huggingface.co/papers/1705.00652))

This section explains how the contrastive training signal is formed
during fine-tuning — the mechanism referred to as "contrastive learning"
in the paper.

**Training data format.** Each training example is a `(query, pos)` pair.
We do NOT explicitly provide negative passages to the trainer. Instead,
negatives are constructed automatically at training time through a
mechanism called **in-batch negatives**.

**In-batch negatives (InfoNCE / MNRL).** Given a batch of `B` pairs
`{(q₁, p₁), (q₂, p₂), ..., (qB, pB)}`, the loss function:

1. Encodes all `B` queries and all `B` passages into dense vectors.
2. Computes a `B × B` cosine similarity matrix between all queries
   and all passages.
3. For each query `qᵢ`, the passage `pᵢ` is the **positive** (correct
   answer), and all other passages `{p₁, ..., pᵢ₋₁, pᵢ₊₁, ..., pB}`
   are treated as **negatives** (wrong answers).
4. Applies cross-entropy loss to push `qᵢ` closer to `pᵢ` and away
   from all other passages in the batch.

```
Similarity matrix (batch_size=4 example):

              p₁     p₂     p₃     p₄
        q₁ [ 0.92   0.45   0.51   0.38 ]  ← maximize (q₁, p₁)
        q₂ [ 0.41   0.89   0.47   0.52 ]  ← maximize (q₂, p₂)
        q₃ [ 0.50   0.43   0.91   0.40 ]  ← maximize (q₃, p₃)
        q₄ [ 0.39   0.48   0.42   0.87 ]  ← maximize (q₄, p₄)

Diagonal = positive pairs (should be highest in each row)
Off-diagonal = in-batch negatives (should be lower)
```

This means that with a batch size of 128, each query has **127 implicit
negatives** per training step — all from the same legal domain, making
them naturally hard negatives.

**Why larger batches improve performance.** More samples per batch =
more in-batch negatives = harder contrastive signal = better
discrimination. This is why `CachedMultipleNegativesRankingLoss` is
valuable: it allows an effective batch size of 128 while only using
the GPU memory of `mini_batch_size=16`, by caching intermediate
embeddings (GradCache; Gao et al., 2021).

**Role of `BatchSamplers.NO_DUPLICATES`.** This sampler ensures that
no two samples in the same batch share identical text (query or
passage). This is critical because:

- If `p₃ == p₇` (duplicate passages in the batch), then `q₃` would
  have its own correct answer appearing as a "negative" — sending a
  contradictory gradient signal to the model.
- `NO_DUPLICATES` prevents this by checking for text duplicates when
  forming each batch.

**Important:** `NO_DUPLICATES` is a **sampler** (controls batch
composition), not a loss function. It does not generate negatives —
the loss function does that via the similarity matrix above.

**Summary of roles:**

| Component | Role |
|-----------|------|
| `CachedMNRL` (loss) | Constructs in-batch negatives from the B×B similarity matrix and computes cross-entropy |
| `NO_DUPLICATES` (sampler) | Ensures no duplicate texts in a batch, preventing false negatives |
| `per_device_train_batch_size=128` | Controls how many in-batch negatives each query sees (127) |
| `mini_batch_size=16` | Controls GPU memory usage (forward pass in chunks of 16) |


## 7. Build Dev Evaluator (monitoring during training)

This evaluator runs every 100 training steps to monitor convergence.
It uses data from the **training region** (first 380K instances), NOT
from the final evaluation region. It is intentionally smaller (50K
queries, 80K corpus) for speed.

In [7]:
queries = dict(enumerate(eval_dataset["query"]))

corpus_list = eval_dataset["pos"][:] + train_dataset.select(range(30_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}

dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-bge-eval-50kq-80kd",
    show_progress_bar=True,
)

# Evaluate base model before training
dev_evaluator(model)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [08:19<08:19, 499.94s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [13:21<00:00, 400.63s/it]


{'legal-spanish-bge-eval-50kq-80kd_cosine_accuracy@1': 0.7689,
 'legal-spanish-bge-eval-50kq-80kd_cosine_accuracy@3': 0.84754,
 'legal-spanish-bge-eval-50kq-80kd_cosine_accuracy@5': 0.87202,
 'legal-spanish-bge-eval-50kq-80kd_cosine_accuracy@10': 0.89908,
 'legal-spanish-bge-eval-50kq-80kd_cosine_precision@1': 0.7689,
 'legal-spanish-bge-eval-50kq-80kd_cosine_precision@3': 0.2825133333333332,
 'legal-spanish-bge-eval-50kq-80kd_cosine_precision@5': 0.174404,
 'legal-spanish-bge-eval-50kq-80kd_cosine_precision@10': 0.08990800000000002,
 'legal-spanish-bge-eval-50kq-80kd_cosine_recall@1': 0.7689,
 'legal-spanish-bge-eval-50kq-80kd_cosine_recall@3': 0.84754,
 'legal-spanish-bge-eval-50kq-80kd_cosine_recall@5': 0.87202,
 'legal-spanish-bge-eval-50kq-80kd_cosine_recall@10': 0.89908,
 'legal-spanish-bge-eval-50kq-80kd_cosine_ndcg@10': 0.8344443566149717,
 'legal-spanish-bge-eval-50kq-80kd_cosine_mrr@10': 0.8137107460317397,
 'legal-spanish-bge-eval-50kq-80kd_cosine_map@100': 0.816272310448278

## 8. Train

In [8]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Legal-spanish-bge-eval-50kq-80kd Cosine Accuracy@1,Legal-spanish-bge-eval-50kq-80kd Cosine Accuracy@3,Legal-spanish-bge-eval-50kq-80kd Cosine Accuracy@5,Legal-spanish-bge-eval-50kq-80kd Cosine Accuracy@10,Legal-spanish-bge-eval-50kq-80kd Cosine Precision@1,Legal-spanish-bge-eval-50kq-80kd Cosine Precision@3,Legal-spanish-bge-eval-50kq-80kd Cosine Precision@5,Legal-spanish-bge-eval-50kq-80kd Cosine Precision@10,Legal-spanish-bge-eval-50kq-80kd Cosine Recall@1,Legal-spanish-bge-eval-50kq-80kd Cosine Recall@3,Legal-spanish-bge-eval-50kq-80kd Cosine Recall@5,Legal-spanish-bge-eval-50kq-80kd Cosine Recall@10,Legal-spanish-bge-eval-50kq-80kd Cosine Ndcg@10,Legal-spanish-bge-eval-50kq-80kd Cosine Mrr@10,Legal-spanish-bge-eval-50kq-80kd Cosine Map@100
100,0.028237,0.016568,0.904460,0.949120,0.961040,0.973180,0.904460,0.316373,0.192208,0.097318,0.904460,0.949120,0.961040,0.973180,0.939929,0.929159,0.930166
200,0.017283,0.012377,0.913620,0.956080,0.967280,0.978200,0.913620,0.318693,0.193456,0.097820,0.913620,0.956080,0.967280,0.978200,0.947164,0.937094,0.937958
300,0.014886,0.011436,0.918860,0.958140,0.968700,0.979360,0.918860,0.319380,0.193740,0.097936,0.918860,0.958140,0.968700,0.979360,0.950157,0.940698,0.941524
400,0.023855,0.010433,0.919840,0.960360,0.969920,0.980580,0.919840,0.320120,0.193984,0.098058,0.919840,0.960360,0.969920,0.980580,0.951451,0.941996,0.942824
500,0.018218,0.009361,0.924500,0.962260,0.971760,0.981480,0.924500,0.320753,0.194352,0.098148,0.924500,0.962260,0.971760,0.981480,0.954172,0.945305,0.946083
600,0.015707,0.008750,0.925380,0.963760,0.972500,0.982740,0.925380,0.321253,0.194500,0.098274,0.925380,0.963760,0.972500,0.982740,0.955285,0.946373,0.947112
700,0.012830,0.008288,0.924300,0.962900,0.972480,0.982520,0.924300,0.320967,0.194496,0.098252,0.924300,0.962900,0.972480,0.982520,0.954511,0.945424,0.946175
800,0.007042,0.007760,0.928500,0.965380,0.974840,0.984400,0.928500,0.321793,0.194968,0.098440,0.928500,0.965380,0.974840,0.984400,0.957515,0.948792,0.949455
900,0.010812,0.007341,0.929340,0.967720,0.976160,0.985900,0.929340,0.322573,0.195232,0.098590,0.929340,0.967720,0.976160,0.985900,0.958970,0.950207,0.950803
1000,0.012054,0.007180,0.932060,0.968320,0.977320,0.986260,0.932060,0.322773,0.195464,0.098626,0.932060,0.968320,0.977320,0.986260,0.960378,0.951959,0.952565


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [02:07<00:00, 63.57s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2344, training_loss=0.01610696719444449, metrics={'train_runtime': 15722.0017, 'train_samples_per_second': 19.082, 'train_steps_per_second': 0.149, 'total_flos': 0.0, 'train_loss': 0.01610696719444449, 'epoch': 1.0})

## 9. Post-Training Evaluation and Save

Running the evaluators after training serves two purposes: (1) confirm
final metrics, and (2) **log results into the model card** — Hugging Face
automatically includes the last evaluation scores in the model card
when pushing to the Hub.

We evaluate on both the **dev set** (50K queries) and a held-out **test
set** (30K queries, 20K corpus) to provide two independent performance
snapshots in the model card.

In [9]:
# Re-run dev evaluator — results are logged into the model card
dev_evaluator(model)

# Test set evaluation (30K queries, 20K corpus) — also logged into model card
queries = dict(enumerate(test_dataset["query"]))
corpus_list = test_dataset["pos"][:] + train_dataset.select(range(20_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}
test_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-bge-test-30kq-50kd",
    show_progress_bar=True,
)
test_evaluator(model)

# Save the trained model
final_output_dir = f"models/{run_name}/final"
model.save_pretrained(final_output_dir)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [01:18<01:18, 78.06s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [01:16<00:00, 76.93s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 10. Push to Hub

In [10]:
try:
    model.push_to_hub(run_name)
except Exception:
    logging.error(
        f"Error uploading model to Hub:\n{traceback.format_exc()}"
        f"Model is saved locally at: {final_output_dir}\n"
        f"To retry: model = SentenceTransformer('{final_output_dir}')\n"
        f"Then: model.push_to_hub('{run_name}')"
    )

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpbtk98yeq/tokenizer.json:  50%|####9     | 8.34MB / 16.8MB            

  ...tk98yeq/model.safetensors:   1%|1         | 24.7MB / 2.27GB            